# Climatology Model Evaluation

Evaluates the trained model on the test split.

For each test simulation, the model is given K-step windows of vorticity and predicts the time-mean zonal-mean zonal wind profile (climatology). Predictions are averaged over all windows from the simulation and compared to the ground-truth climatology computed from the full run.

**Plots produced**
1. Training loss curve
2. Truth vs prediction profiles (small multiples per sim)
3. Error profiles by latitude
4. Scatter: predicted vs truth at each latitude
5. Prediction variance across windows (consistency check)

In [ ]:
import csv
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

ML_DIR = Path("..")
sys.path.insert(0, str(ML_DIR / "src"))

from ml.config import load as load_config
from ml.data.climatology import is_climatology_var
from ml.data.splits import Splits
from ml.data.isca_segment import aggregated_read_field, read_segment
from ml.diagnostics import find_spinup_time, find_zonal_mean_convergence_time
from ml.diagnostics.enstrophy import mean_enstrophy
from ml.diagnostics.convergence import compute_time_mean_zonal_mean
from ml.diagnostics.spatial import cosine_latitude_weights
from ml.training.model import build_model

plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Parameters ---
CONFIG_PATH = ML_DIR / "configs" / "default.toml"
TRAINING_DIR = None  # None = derive from config (experiment_dir/training)

In [ ]:
cfg = load_config(CONFIG_PATH)
training_dir = TRAINING_DIR or cfg.paths.training_dir
assert training_dir.exists(), f"training dir not found: {training_dir}"

splits = Splits.from_config(cfg.paths)
print(f"train: {len(splits.train)}  val: {len(splits.validation)}  test: {len(splits.test)} simulations")

## Training loss curve

In [ ]:
metrics_file = training_dir / "epoch-metrics.csv"
assert metrics_file.exists(), f"epoch-metrics.csv not found: {metrics_file}"

epochs, train_losses, val_losses = [], [], []
with open(metrics_file) as f:
    for row in csv.DictReader(f):
        epochs.append(int(row["epoch"]))
        train_losses.append(float(row["train_loss"]))
        v = row.get("val_loss", "")
        val_losses.append(float(v) if v else float("nan"))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(epochs, train_losses, label="train")
val_arr = np.array(val_losses)
if not np.all(np.isnan(val_arr)):
    ax.plot(epochs, val_arr, label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Training loss curve")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Load model

In [ ]:
clim_target = all(is_climatology_var(v) for v in cfg.data.y_vars)
model = build_model(
    cfg.model,
    dropout=cfg.training.regularization.dropout,
    zonal_mean=clim_target,
)

params_path = training_dir / "parameters.pt"
assert params_path.exists(), f"model parameters not found: {params_path}"
state_dict = torch.load(params_path, map_location="cpu", weights_only=False)
state_dict.pop("_metadata", None)
model.load_state_dict(state_dict)
model.eval()

norm_path = training_dir / "normalization.pt"
assert norm_path.exists(), f"normalization stats not found: {norm_path}"
norm = torch.load(norm_path, map_location="cpu", weights_only=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"architecture: {cfg.model.architecture}  params: {n_params:,}  zonal_mean_wrapper: {clim_target}")

## Evaluate test simulations

For each test simulation:
- Detect spinup time $t_s$ (enstrophy stabilisation) and convergence time $t_c$ (zonal-mean $u$ convergence)
- Compute ground-truth climatology: $\langle u \rangle_{t_c:T}$ (time-mean zonal-mean)
- Extract K-step vorticity windows starting from $t_s$ with the training stride
- Run each window through the model; average predictions

In [ ]:
def predict_windows(model, norm, vor_windows: np.ndarray) -> np.ndarray:
    """vor_windows: (N, K, lat, lon) -> (N, lat) predicted climatology"""
    x = torch.from_numpy(vor_windows.astype(np.float32))
    with torch.no_grad():
        x_norm = (x - norm["x_mean"]) / norm["x_std"]
        y_norm = model(x_norm)                      # (N, 1, lat)
        y = y_norm * norm["y_std"] + norm["y_mean"]
    return y.squeeze(1).numpy()                     # (N, lat)


def extract_windows(vor_full: np.ndarray, t_s: int, K: int, stride: int) -> np.ndarray:
    T = vor_full.shape[0]
    windows = [vor_full[t : t + K] for t in range(t_s, T - K + 1, stride)]
    if not windows:
        return np.empty((0, K, *vor_full.shape[1:]), dtype=vor_full.dtype)
    return np.stack(windows, axis=0)


def evaluate_sim(sim_dir: Path) -> dict | None:
    nc_files = sorted(sim_dir.glob(cfg.data.segment_pattern))
    if not nc_files:
        return None

    ds_cache: dict = {}
    try:
        lat = read_segment(nc_files[0], ds_cache)["lat"].values.astype(np.float64)
        vor_full = aggregated_read_field(nc_files, "vor", ds_cache)
        u_full = aggregated_read_field(nc_files, "ucomp", ds_cache)
    finally:
        for d in ds_cache.values():
            d.close()

    T = vor_full.shape[0]

    t_s = 0
    if cfg.data.spinup is not None:
        detected = find_spinup_time(mean_enstrophy(vor_full, lat), **cfg.data.spinup.to_kwargs())
        if detected is not None:
            t_s = detected

    t_c = t_s
    if cfg.data.convergence is not None:
        detected = find_zonal_mean_convergence_time(
            u_full, lat, t_s,
            threshold=cfg.data.convergence.threshold,
            hold=cfg.data.convergence.hold,
        )
        if detected is not None:
            t_c = detected

    truth = compute_time_mean_zonal_mean(u_full, t_c, T)   # (lat,)

    K = cfg.data.windows.length
    stride = cfg.data.windows.stride
    windows = extract_windows(vor_full, t_s, K, stride)
    if len(windows) == 0:
        return None

    preds = predict_windows(model, norm, windows)           # (N, lat)

    return {
        "sim_dir": sim_dir,
        "lat": lat,
        "truth": truth,
        "predictions": preds,
        "mean_pred": preds.mean(axis=0),
        "t_s": t_s,
        "t_c": t_c,
        "n_windows": len(windows),
    }

In [ ]:
from tqdm.notebook import tqdm

results = []
for sim_dir in tqdm(splits.test, desc="test sims"):
    res = evaluate_sim(sim_dir)
    if res is not None:
        results.append(res)

print(f"evaluated {len(results)} / {len(splits.test)} test simulations")

## Metrics

In [ ]:
def rel_l2(pred, truth, lat):
    w = cosine_latitude_weights(lat)
    return float(np.sqrt(np.sum(w * (pred - truth) ** 2) / np.sum(w * truth ** 2)))


def rmse(pred, truth, lat):
    w = cosine_latitude_weights(lat)
    return float(np.sqrt(np.sum(w * (pred - truth) ** 2) / np.sum(w)))


header = f"  {'simulation':<32} {'windows':>8} {'t_s':>6} {'t_c':>6} {'rmse':>10} {'relL2':>10}"
print(header)
print("  " + "-" * (len(header) - 2))
all_rmse, all_rel = [], []
for res in results:
    lat = res["lat"]
    r = rmse(res["mean_pred"], res["truth"], lat)
    rl = rel_l2(res["mean_pred"], res["truth"], lat)
    all_rmse.append(r)
    all_rel.append(rl)
    print(f"  {res['sim_dir'].name:<32} {res['n_windows']:>8} {res['t_s']:>6} {res['t_c']:>6} {r:>10.4f} {rl:>10.4f}")
print("  " + "-" * (len(header) - 2))
print(f"  {'mean':<32} {'':>8} {'':>6} {'':>6} {np.mean(all_rmse):>10.4f} {np.mean(all_rel):>10.4f}")

## Truth vs prediction profiles

In [ ]:
n = len(results)
ncols = min(4, n)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 2.8, nrows * 3.2),
    sharex=True, sharey=True,
    constrained_layout=True,
)
axes_flat = list(np.array(axes).flat) if n > 1 else [axes]

for i, res in enumerate(results):
    ax = axes_flat[i]
    lat = res["lat"]
    for pred in res["predictions"]:
        ax.plot(pred, lat, color="tab:blue", alpha=0.08, linewidth=0.6)
    ax.plot(res["mean_pred"], lat, color="tab:blue", linewidth=1.5, label="pred (mean)")
    ax.plot(res["truth"], lat, color="tab:orange", linestyle="--", linewidth=1.5, label="truth")
    ax.axvline(0.0, color="k", linewidth=0.4, alpha=0.4)
    rl = rel_l2(res["mean_pred"], res["truth"], lat)
    ax.set_title(res["sim_dir"].name, fontsize=8)
    ax.set_xlabel(f"u [m/s]  relL2={rl:.3f}", fontsize=8)
    if i % ncols == 0:
        ax.set_ylabel("lat [deg]")
    if i == 0:
        ax.legend(fontsize=8)

for ax in axes_flat[n:]:
    ax.set_visible(False)

fig.suptitle("Climatology: truth (orange) vs model prediction (blue, faded = individual windows)")
plt.show()

## Error profiles by latitude

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for res in results:
    err = res["mean_pred"] - res["truth"]
    ax.plot(err, res["lat"], alpha=0.7, linewidth=1.2, label=res["sim_dir"].name)
ax.axvline(0.0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("error  (pred - truth) [m/s]")
ax.set_ylabel("lat [deg]")
ax.set_title("Prediction error by latitude per test simulation")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

## Scatter: predicted vs truth

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for res in results:
    ax.scatter(res["truth"], res["mean_pred"], s=10, alpha=0.6, label=res["sim_dir"].name)
all_truth = np.concatenate([r["truth"] for r in results])
lo, hi = all_truth.min(), all_truth.max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.8, label="y=x")
ax.set_xlabel("truth [m/s]")
ax.set_ylabel("prediction [m/s]")
ax.set_title("Truth vs prediction at each latitude (all test sims)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Prediction variance across windows

Low std = the model gives consistent predictions regardless of the input window. High std at a latitude = the model is uncertain there.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for res in results:
    std = res["predictions"].std(axis=0)
    ax.plot(std, res["lat"], alpha=0.7, linewidth=1.2, label=res["sim_dir"].name)
ax.set_xlabel("prediction std across windows [m/s]")
ax.set_ylabel("lat [deg]")
ax.set_title("Within-simulation prediction variance (lower = more consistent)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()